### **Environment Setup & Package Installation**

In [ ]:
%%capture
!pip install -q boxmot trackeval opencv-python numpy pandas huggingface_hub ultralytics

from kaggle_secrets import UserSecretsClient 
gh_token = UserSecretsClient().get_secret("GH_TOKEN")
!git clone https://{gh_token}@github.com/Hannysia/drone_detection.git
!pip install -e drone_detection/

%cd /kaggle/working/drone_detection

%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
import pandas as pd
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi
from IPython.display import display

from src.config import (
    HF_TOKEN, REPO_ID, LOCAL_HF_DATA_DIR, MOT_DATASET_DIR, SEQMAPS_DIR, WORKING_DIR, 
    YOLO_DETECTIONS_OUT, YOLO_MODEL_PATH, TRACKERS_OUT_DIR, VIDEO_EXTENSIONS
)
from src.processing.dataset import compile_kaggle_mot_dataset
from src.processing.tracking import run_tracking_pipeline, run_yolo_inference_for_tracking
from src.processing.evaluation import run_trackeval, parse_trackeval_summary, find_and_display_best_configs

BYTETRACK_GRID = {
    "track_thresh": [0.3, 0.4, 0.5, 0.6, 0.7],
    "track_buffer": [15, 30, 45, 60],
    "match_thresh": [0.7, 0.8, 0.9]
}

OCSORT_GRID = {
    "det_thresh": [0.3, 0.4, 0.5, 0.6, 0.7],
    "max_age": [15, 30, 45, 60],
    "iou_threshold": [0.2, 0.3, 0.4, 0.5]
}

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### **Asset Synchronization & MOT Dataset Compiling**

In [ ]:
print("Downloading required folders from Hugging Face...")
LOCAL_HF_DATA_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN,
    local_dir=str(LOCAL_HF_DATA_DIR),
    allow_patterns=["test/**", "val/**", "labels/**", "models/yolo26n_final_model_best.engine"]
)
print("Download from Hugging Face complete!")

In [ ]:
compile_kaggle_mot_dataset(LOCAL_HF_DATA_DIR, MOT_DATASET_DIR, SEQMAPS_DIR, tuple(VIDEO_EXTENSIONS))

### **Grid Search Calibration (Ground Truth Detections)**

In [ ]:
import itertools

# --- 1. Run ByteTrack Grid ---
bytetrack_runs = []
keys, values = zip(*BYTETRACK_GRID.items())
bytetrack_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print("Starting ByteTrack Grid Search...")
for idx, config in enumerate(bytetrack_combinations):
    run_name = f"bytetrack_run_{idx}"
    bytetrack_runs.append(run_name)
    run_tracking_pipeline(
        split_name="val", 
        tracker_name="bytetrack", 
        tracker_config=config, 
        run_dir_name=run_name,
        dataset_base_dir=MOT_DATASET_DIR,
        trackers_out_dir=TRACKERS_OUT_DIR
    )

run_trackeval(
    split_name="val", 
    trackers_to_eval=bytetrack_runs,
    dataset_base_dir=MOT_DATASET_DIR,
    trackers_out_dir=TRACKERS_OUT_DIR
)


# --- 2. Run OC-SORT Grid ---
ocsort_runs = []
keys, values = zip(*OCSORT_GRID.items())
ocsort_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print("\nStarting OC-SORT Grid Search...")
for idx, config in enumerate(ocsort_combinations):
    run_name = f"ocsort_run_{idx}"
    ocsort_runs.append(run_name)
    run_tracking_pipeline(
        split_name="val", 
        tracker_name="ocsort", 
        tracker_config=config, 
        run_dir_name=run_name,
        dataset_base_dir=MOT_DATASET_DIR,
        trackers_out_dir=TRACKERS_OUT_DIR
    )

run_trackeval(
    split_name="val", 
    trackers_to_eval=ocsort_runs,
    dataset_base_dir=MOT_DATASET_DIR,
    trackers_out_dir=TRACKERS_OUT_DIR
)

In [ ]:
best_bytetrack_params, best_ocsort_params = find_and_display_best_configs(
    trackers_out_dir=TRACKERS_OUT_DIR,
    bytetrack_combinations=bytetrack_combinations,
    ocsort_combinations=ocsort_combinations,
    split_name="val",
    suffix=""
)


                 FINAL COMPARATIVE METRICS REPORT (BEST CALIBRATED CONFIGURATIONS)


Tracker,Conf Thresh,Memory (Frames),IoU Thresh,HOTA,DetA,AssA,IDF1,MOTA,MOTP,FP,FN,IDSW,Frag
ByteTrack,0.300000,30,0.900000,85.92%,85.25%,86.98%,96.23%,97.80%,88.41%,71,92,11,14
OC-SORT,0.300000,30,0.200000,96.87%,98.44%,95.33%,96.53%,98.32%,100.00%,0,123,10,0


In [18]:
print("Running final Stage 1 test with best calibrated parameters...")

# Run optimal ByteTrack
run_tracking_pipeline(
    split_name="test", 
    tracker_name="bytetrack", 
    tracker_config=best_bytetrack_params, 
    run_dir_name="bytetrack_final"
)

# Run optimal OC-SORT
run_tracking_pipeline(
    split_name="test", 
    tracker_name="ocsort", 
    tracker_config=best_ocsort_params, 
    run_dir_name="ocsort_final"
)

# Run evaluation on test split
run_trackeval(split_name="test", trackers_to_eval=["bytetrack_final", "ocsort_final"])

Running final Stage 1 test with best calibrated parameters...
===> [BYTETRACK] BYTETRACK_FINAL | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.7} | Avg Latency: 0.568 ms/frame | Avg FPS: 1759.4
===> [OCSORT] OCSORT_FINAL | Config: {'det_thresh': 0.3, 'max_age': 15, 'iou_threshold': 0.2} | Avg Latency: 0.577 ms/frame | Avg FPS: 1731.6

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          
Evaluating 2 configurations on split 'test'...

Evaluating 2 tracker(s) on 84 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating bytetrack_final


Evaluating ocsort_final

Evaluation complete. Results saved to disk successfully.


In [ ]:
bt_test_metrics = parse_trackeval_summary(TRACKERS_OUT_DIR / "drone-test" / "bytetrack_final" / "pedestrian_summary.txt")
oc_test_metrics = parse_trackeval_summary(TRACKERS_OUT_DIR / "drone-test" / "ocsort_final" / "pedestrian_summary.txt")

df_test = pd.DataFrame([bt_test_metrics, oc_test_metrics], index=["ByteTrack (Final Test)", "OC-SORT (Final Test)"])
display(df_test)


,HOTA,DetA,AssA,DetRe,DetPr,AssRe,AssPr,LocA,OWTA,HOTA(0),LocA(0),HOTALocA(0),MOTA,MOTP,MODA,CLR_Re,CLR_Pr,MTR,PTR,MLR,CLR_TP,CLR_FN,CLR_FP,IDSW,MT,PT,ML,Frag,sMOTA,IDF1,IDR,IDP,IDTP,IDFN,IDFP,Dets,GT_Dets,IDs,GT_IDs
ByteTrack (Final Test),83.550,84.775,82.639,88.904,89.655,85.285,92.282,89.451,85.707,95.316,87.691,83.584,97.394,87.975,97.842,98.502,99.334,98.824,1.1765,0.0,8352.0,127.0,56.0,38.0,84.0,1.0,0.0,15.0,85.549,93.350,92.959,93.744,7882.0,597.0,526.0,8408.0,8479.0,121.0,85.0
OC-SORT (Final Test),96.812,98.585,95.071,98.585,100.000,95.071,100.000,100.000,96.812,96.812,100.000,96.812,98.455,100.000,98.585,98.585,100.000,100.000,0.0000,0.0,8359.0,120.0,0.0,11.0,85.0,0.0,0.0,0.0,98.455,96.722,96.037,97.416,8143.0,336.0,216.0,8359.0,8479.0,96.0,85.0


### **YOLO Inference & Detection Syncing**

In [ ]:
run_yolo_inference_for_tracking(YOLO_MODEL_PATH, SEQMAPS_DIR, LOCAL_HF_DATA_DIR, YOLO_DETECTIONS_OUT, tuple(VIDEO_EXTENSIONS))


--- Running YOLO26 GPU inference on 'val' split ---
Inference on: delta_thermal_field_136.mp4
Loading /kaggle/working/drone_dataset/models/yolo26n_final_model_best.engine for TensorRT inference...
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.2.0'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 39.59s
Prepared 3 packages in 19.60s
Installed 3 packages in 8ms
 + tensorrt-cu12==11.1.0.106
 + tensorrt-cu12-bindings==11.1.0.106
 + tensorrt-cu12-libs==11.1.0.106

requirements: AutoUpdate success ✅ 60.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

[06/22/2026-12:34:36] [TRT] [I] Loaded engine size: 6 MiB
[06/22/2026-12:34:36] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +52, now: CPU 0, GPU 60 (MiB)
Inference on: delta_thermal_sky_229.mp4
Inference on: wing_rgb_sky_335.mp4
Inference on: wing_rgb_sky_336.m4v
Inference on: win

In [ ]:
# 1. Upload to HF
api = HfApi(token=HF_TOKEN)
api.upload_folder(folder_path=str(YOLO_DETECTIONS_OUT), path_in_repo="yolo_detections", repo_id=REPO_ID, repo_type="dataset")

# 2. Download back from HF
YOLO_VAL_DETECTIONS = WORKING_DIR / "yolo_detections" / "val"
YOLO_VAL_DETECTIONS.parent.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN, local_dir=str(WORKING_DIR), allow_patterns=["yolo_detections/**"])

Fetching ... files: 0it [00:00, ?it/s]

Download from Hugging Face complete.


### **Grid Search Calibration (YOLO Model Detections)**

In [ ]:
bytetrack_runs_model = []
print("Starting ByteTrack Grid Search on YOLO26 detections...")
for idx, config in enumerate(bytetrack_combinations):
    run_name = f"bytetrack_model_run_{idx}"
    bytetrack_runs_model.append(run_name)
    run_tracking_pipeline(
        split_name="val", tracker_name="bytetrack", tracker_config=config, 
        run_dir_name=run_name, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR,
        detections_source_dir=YOLO_VAL_DETECTIONS
    )

run_trackeval(split_name="val", trackers_to_eval=bytetrack_runs_model, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR)

ocsort_runs_model = []
print("\nStarting OC-SORT Grid Search on YOLO26 detections...")
for idx, config in enumerate(ocsort_combinations):
    run_name = f"ocsort_model_run_{idx}"
    ocsort_runs_model.append(run_name)
    run_tracking_pipeline(
        split_name="val", tracker_name="ocsort", tracker_config=config, 
        run_dir_name=run_name, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR,
        detections_source_dir=YOLO_VAL_DETECTIONS
    )

run_trackeval(split_name="val", trackers_to_eval=ocsort_runs_model, dataset_base_dir=MOT_DATASET_DIR, trackers_out_dir=TRACKERS_OUT_DIR)

In [ ]:
best_bytetrack_model, best_ocsort_model = find_and_display_best_configs(
    trackers_out_dir=TRACKERS_OUT_DIR,
    bytetrack_combinations=bytetrack_combinations,
    ocsort_combinations=ocsort_combinations,
    split_name="val",
    suffix="_model"
)


                 YOLO26 DETECTIONS COMPARATIVE METRICS REPORT (BEST CONFIGURATIONS)


Tracker,Conf Thresh,Memory (Frames),IoU Thresh,HOTA,DetA,AssA,IDF1,MOTA,MOTP,FP,FN,IDSW,Frag
ByteTrack,0.600000,15,0.900000,69.26%,66.49%,72.68%,84.14%,77.29%,83.86%,628,1134,34,39
OC-SORT,0.400000,60,0.200000,61.38%,65.77%,57.43%,69.11%,69.99%,89.92%,397,1861,115,18
